In [23]:
from ase.io import read, write
from ase.visualize import view
from rdkit.Chem import Draw

import os
import yaml

from openmm.openmm import XmlSerializer
from openmm.app import PME
from openmm.app.forcefield import ForceField
from openmm.app.pdbfile import PDBFile
from openmmforcefields.generators import (
    GAFFTemplateGenerator,
)

# from molinfo import asemol2molinfo, il_assign, get_smiles_from_molinfo, get_sdf_from_molinfo, save_molinfo
# from molinfo import load_geoopt_results, get_charges_from_molinfo
from imolcryff.asemol import asemol_wrapper, aseatoms2pdb, merge_asemols
from imolcryff.molinfo import Mol_Info
from imolcryff.g16wrapper import molinfo_setg16opt, molinfo_setg16charge, molinfo_setg16dihedral
from imolcryff.gaffil_generators import GAFFilTemplateGenerator

In [ ]:
!pwd

In [ ]:
params = yaml.safe_load(open('all_params.yaml'))
params

In [ ]:
!ls

In [6]:
atoms = read("ordered_Li(FSA)(C2)2_233K.cif")
asemol_wrap = asemol_wrapper(atoms)
atoms_unwrap = asemol_wrap.unwrap_molecules()
atomslist_mols, molecule_list = asemol_wrap.get_ase_molecules(ordered=True)

merged_aseatoms = merge_asemols(atomslist_mols)
aseatoms2pdb("merged.pdb", merged_aseatoms)

In [ ]:
import rdkit
rdkit.__version__

In [8]:
mol_info = Mol_Info()

In [9]:
for i, mol_idx in enumerate(molecule_list):
    al = [atomslist_mols[i] for i in mol_idx]
    mol_info.append_fromAtomsList(al, key=f"MOL_{i}", Nmols = len(mol_idx))

In [ ]:
mol_info.auto_charge_assign()
mol_info.get_rdkitmol()
mol_info.il_assign()
rdkitmol_list = [mol_info.mol_info[key]["rdkitmol2d"] for key in mol_info.mol_info.keys()]
Draw.MolsToGridImage(rdkitmol_list, molsPerRow = 3, subImgSize = (300,150),
                     legends = [molkey for molkey in mol_info.mol_info.keys()], maxMols = 60)

In [11]:
default_params_dihedral = {
    "method": "wb97xd",
    "basis": "6-311++g(d,p)",
    "opt": "modredundant",
    "mem": "92GB",
    "nprocshared": 40,
    "addsec": "hoge-hoge"
}

In [ ]:
import imolcryff

imolcryff.__file__

In [ ]:
mol_info.mol_info

In [14]:
mol_info.get_smiles_from_molinfo()
mol_info.get_sdf_from_molinfo()
molinfo_setg16opt(mol_info.mol_info, params["g16opt"])
# mol_info.do_geoopt()
mol_info.load_geoopt_results()

molinfo_setg16charge(mol_info.mol_info, params["charge"])
# mol_info.do_charge()

# molinfo_setg16dihedral(mol_info.mol_info)
# mol_info.do_dihedral()

In [ ]:
for i in range(3):
    try:
        _ = mol_info.get_charges_from_molinfo()
    except:
        pass

In [ ]:
mol_info.adjust_charges(params["ff_params"])

In [ ]:
mmm = mol_info.get_molecule_omm()

if params["ff_params"]["fftype"].split("-")[0] == "gaff":
    gaff = GAFFilTemplateGenerator(molecules=mmm, forcefield=params["ff_params"]["fftype"])
    # gaff = GAFFTemplateGenerator(molecules=mmm, forcefield="gaff-2.11")

# for i in range(len(mmm)):
#     ffxml = gaff.generate_residue_template(mmm[i])
#     with open(f"gaffxml_{i}.xml", "w") as f:
#         f.write(ffxml)
# import openmmforcefields
# import shutil
# ff_dir = os.path.dirname(openmmforcefields.__file__)
# ff_xml = os.path.join(ff_dir, "ffxml", params["ff_params"]["iontype"])
# shutil.copy(ffxml,  "./")

forcefield = ForceField(params["ff_params"]["iontype"])
forcefield.registerTemplateGenerator(gaff.generator)

In [18]:
# forcefield._templates["LI"].atoms[0].parameters["charge"] = 0.444
# Adjust Single-atom ion charge
for key in mol_info.mol_info.keys():
    natoms = mol_info.mol_info[key]["rdkitmol"].GetNumAtoms()
    if natoms == 1:
        symbol = mol_info.mol_info[key]["rdkitmol"].GetAtoms()[0].GetSymbol()
        for elem in forcefield._templates.keys():
            p = forcefield._templates[elem]
            for at in p.atoms:
                if at.element.symbol == symbol:
                    at.parameters["charge"] = mol_info.mol_info[key]["charges"][0]

In [19]:
# pdb = PDBFile("merged_444_woLi.pdb")
# a_cif = read("merged_444_woLi.pdb")

pdb = PDBFile("merged.pdb")
a_cif = read("merged.pdb")
pdb_b = asemol_wrapper(a_cif)
_ = pdb_b.get_bonds()
bonds_list = [ [bond[0], bond[1]] for bond in pdb_b.bonds]
atomlist_openmm = [a for a in pdb.topology.atoms()]
for bond in bonds_list:
    a1 = atomlist_openmm[bond[0]]
    a2 = atomlist_openmm[bond[1]]
    pdb.topology.addBond(a1, a2)

In [20]:
sys = forcefield.createSystem(pdb.topology, nonbondedMethod=PME)
with open('system.xml', 'w') as output:
    output.write(XmlSerializer.serialize(sys))

In [21]:
from openmm import app
from parmed.openmm import load_topology

parm_top = load_topology(topology=pdb.topology, system=sys, xyz=pdb.positions)

parm_top.save('test.top', overwrite=True)
parm_top.save('test.gro', overwrite=True)

#################
##### Error #####
#################
# from openff.toolkit import Topology
# from openff.interchange.interop import openmm

# topology_off = Topology.from_openmm(pdb.topology, unique_molecules=mmm)
# os.environ["INTERCHANGE_EXPERIMENTAL"] = "1"
# a = openmm.from_openmm(system=sys, topology=topology_off, positions=topology_off.get_positions())
# a.to_prmtop("test.prmtop")
# a.to_inpcrd("test.inpcrd")


In [ ]:
from openff.interchange.components.mdconfig import MDConfig
from openff.toolkit import Topology
from openff.interchange.interop import openmm
from openff.interchange.interop.lammps.export import to_lammps
from imolcryff.exporter_lmp import to_lammps_non_rectangular

topology_off = Topology.from_openmm(pdb.topology, unique_molecules=mmm)
os.environ["INTERCHANGE_EXPERIMENTAL"] = "1"
a = openmm.from_openmm(system=sys, topology=topology_off, positions=pdb.getPositions())
to_lammps(a, "test.data")
# non-rectangular box
# to_lammps_non_rectangular(a, "test.data")

In [23]:
for i in range(len(mmm)):
    ffxml = gaff.generate_residue_template(mmm[i])
    with open(f"gaffxml_{i}.xml", "w") as f:
        f.write(ffxml)

In [24]:
# from openff.interchange.components.mdconfig import MDConfig

# cc = MDConfig.from_interchange(a)
# cc.write_lammps_input(
#             interchange=a,
#             input_file="tmp.in",
#         )

In [25]:
# from openff.toolkit import ForceField, Molecule, Topology
# from openff.units import unit
# from openff.interchange import Interchange
# from openff.interchange.components._packmol import UNIT_CUBE, pack_box
# from openff.interchange.components.mdconfig import MDConfig

# #セルの初期密度（ボックスサイズに反映される）
# density = 0.2
# #中に入れる分子のSMILESと個数のリスト
# # [ ["分子1", 100], ["分子2", 50] ] のような指定も可能
# components = [ ["CCO", 100] ]
# #Minimizeをやる場合はTrue
# minimization = True
# #出力するファイル名
# name_in   = "auto_generated.in"
# name_data = "out.lmp"


# molecules = []
# number_of_components = []
# for component in components:
#     molecules.append(Molecule.from_smiles(component[0]))
#     number_of_components.append(component[1])

# topology = pack_box(
#     molecules=molecules,
#     number_of_copies=number_of_components,
#     mass_density=density * 1e3 * unit.kilogram / unit.meter**3,
#     box_shape=UNIT_CUBE,
# )

# print("Box vector          :", topology.box_vectors)
# print("Number of molecules :", topology.n_molecules)

# force_field = ForceField("openff_unconstrained-2.2.0.offxml")

# interchange = Interchange.from_smirnoff(force_field=force_field, topology=topology)

# if minimization:
#     interchange.minimize()

# interchange.to_lammps(name_data)
# mdconfig = MDConfig.from_interchange(interchange)
# mdconfig.write_lammps_input(input_file=name_in, interchange=interchange)
# print("LAMMPS data file    :", name_data)
# print("LAMMPS in file      :", name_in)


In [ ]:
# from openff.toolkit import Molecule
# 
# molecule = Molecule.from_smiles("c1ccccc1")
# # Create the GAFF template generator
# from openmmforcefields.generators import (
#     GAFFTemplateGenerator,
# )
# 
# gaff = GAFFTemplateGenerator(molecules=molecule)
# from openmm.app import ForceField
# 
# forcefield = ForceField()
# # Register the GAFF template generator
# forcefield.registerTemplateGenerator(gaff.generator)
# # You can now parameterize an OpenMM Topology object that contains the specified molecule.
# # forcefield will load the appropriate GAFF parameters when needed, and antechamber
# # will be used to generate small molecule parameters on the fly.
# import openmm
# 
# system = forcefield.createSystem(
#     topology=molecule.to_topology().to_openmm(),
# )
# with open('system.xml', 'w') as output:
#     output.write(XmlSerializer.serialize(sys))

In [ ]:
# pdb = PDBFile("merged.pdb")
# pdb.topology
# # a_cif = read("merged_444.pdb")
# a_cif = read("merged.pdb")
# pdb_b = asemol_wrapper(a_cif)
# _ = pdb_b.get_bonds()
# bonds_list = [ [bond[0], bond[1]] for bond in pdb_b.bonds]
# atomlist_openmm = [a for a in pdb.topology.atoms()]
# for bond in bonds_list:
#     a1 = atomlist_openmm[bond[0]]
#     a2 = atomlist_openmm[bond[1]]
#     pdb.topology.addBond(a1, a2)

In [ ]:
# from openff.toolkit import Molecule
# import openmm
# from openmm.openmm import XmlSerializer
# from openmm.app import PME

# mmm = mol_info.get_molecule_omm()

# # Create the SMIRNOFF template generator with the default installed force field (openff-2.1.0)
# from openmmforcefields.generators import (
#     SMIRNOFFTemplateGenerator,
# )

# smirnoff = SMIRNOFFTemplateGenerator(molecules=mmm)
# # Create an OpenMM ForceField object with AMBER ff14SB and TIP3P with compatible ions
# from openmm.app import ForceField
# from openmm.app.pdbfile import PDBFile


# forcefield = ForceField(
#     "amber/protein.ff14SB.xml",
#     "amber/tip3p_standard.xml",
#     "amber/tip3p_HFE_multivalent.xml",
# )
# # Register the SMIRNOFF template generator
# forcefield.registerTemplateGenerator(smirnoff.generator)

# # create a System with the non-bonded settings of mainline OpenFF force fields
# # (9 Angstrom cut-off, switching distance applied at 8 Angstrom)

# pdb = PDBFile("merged.pdb")
# pdb.topology
# # a_cif = read("merged_444.pdb")
# a_cif = read("merged.pdb")
# pdb_b = asemol_wrapper(a_cif)
# _ = pdb_b.get_bonds()
# bonds_list = [ [bond[0], bond[1]] for bond in pdb_b.bonds]
# atomlist_openmm = [a for a in pdb.topology.atoms()]
# for bond in bonds_list:
#     a1 = atomlist_openmm[bond[0]]
#     a2 = atomlist_openmm[bond[1]]
#     pdb.topology.addBond(a1, a2)

# system = forcefield.createSystem(pdb.topology, nonbondedMethod=PME)

# with open('system.xml', 'w') as output:
#     output.write(XmlSerializer.serialize(system))

# from openff.toolkit import Topology
# from openff.interchange.interop import openmm


# topology_off = Topology.from_openmm(pdb.topology, unique_molecules=mmm)

# import os
# os.environ["INTERCHANGE_EXPERIMENTAL"] = "1"

# a = openmm.from_openmm(system=system, topology=topology_off, positions=pdb.positions)
# a.to_gromacs("test")

